GEETA 4B — nightly emails (2026-10-13).

* **Parents:** one email per child who used GEETA since the last nightly run, with a daily PDF
  (sessions, skills, slips to practise, progress so far). Nothing is sent on days a child didn't use
  GEETA, or if the parent switched "Email me a daily report" off.
* **Admins:** a technical PDF — service health, usage, exact-Python vs model share, hallucination-gate
  blocks, leak-guard blocks, log errors, security events, backups, disk.
* **Transport:** SMTP from `config/credentials.local.yaml` (git-ignored, mode 600; the owner pastes the
  password). Until it is enabled every email is written to `outputs/email_outbox/` instead.
* Scheduled at 22:30 Adelaide time by the `com.geeta.nightly` LaunchAgent (`deploy/install_services.sh`).


## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import io
import json
import mimetypes
import os
import re
import shutil
import smtplib
import ssl
import subprocess
import time
import urllib.request
from datetime import datetime
from email.message import EmailMessage
from pathlib import Path
from typing import Optional
from xml.sax.saxutils import escape as _x

import yaml

ROOT = Path(globals().get("__nb_path__", "")).resolve().parent.parent if globals().get("__nb_path__") else Path.cwd()
if not (ROOT / "tools").exists():
    ROOT = Path.cwd() if (Path.cwd() / "tools").exists() else Path.cwd().parent

# ---------------------------------------------------------------------------
# Mail settings: config/credentials.local.yaml (git-ignored, mode 600). The
# owner fills in the password; GEETA never prints or logs it. Until "enabled:
# true" + a password are set, every email goes to outputs/email_outbox/ instead.
# ---------------------------------------------------------------------------
MAIL_TEMPLATE = """# GEETA email settings - PRIVATE (never commit or share this file).
# Gmail: turn on 2-Step Verification for the Google account, then create an
# "App password" (Google Account > Security > App passwords) and paste it below.
email:
  enabled: false            # set true once the password is filled in
  smtp_host: smtp.gmail.com
  smtp_port: 587            # 587 = STARTTLS, 465 = SSL
  username: ""              # e.g. geeta.reports@gmail.com
  password: ""              # the 16-letter app password - NOT your normal password
  from_address: ""          # usually the same as username
  from_name: "GEETA maths buddy"
  admin_recipients: []      # extra addresses for the technical report (admins get it anyway)
  send_time: "22:30"        # nightly, 24-hour local time (Mac is on Adelaide time)
"""


def mail_settings_path() -> Path:
    return ROOT / "config" / "credentials.local.yaml"


def load_mail_settings() -> dict:
    p = mail_settings_path()
    if not p.exists():
        p.write_text(MAIL_TEMPLATE)
        p.chmod(0o600)
    data = yaml.safe_load(p.read_text()) or {}
    m = dict(data.get("email") or {})
    m.setdefault("enabled", False)
    m.setdefault("smtp_port", 587)
    m.setdefault("from_name", "GEETA maths buddy")
    m.setdefault("admin_recipients", [])
    m.setdefault("send_time", "22:30")
    m["ready"] = bool(m.get("enabled") and m.get("smtp_host") and m.get("username")
                      and m.get("password") and (m.get("from_address") or m.get("username")))
    return m


def public_mail_status() -> dict:
    """What the admin page may show - never the password."""
    m = load_mail_settings()
    return {"ready": m["ready"], "enabled": bool(m.get("enabled")), "smtp_host": m.get("smtp_host"),
            "from_address": m.get("from_address") or m.get("username") or "",
            "password_set": bool(m.get("password")), "send_time": m.get("send_time"),
            "admin_recipients": m.get("admin_recipients") or [],
            "settings_file": str(mail_settings_path().relative_to(ROOT))}


class Mailer:
    def __init__(self, settings: dict | None = None):
        self.m = settings or load_mail_settings()
        # GEETA_OUTBOX_DIR: tests use a throwaway outbox, never the real one
        self.outbox = Path(os.environ["GEETA_OUTBOX_DIR"]) if os.environ.get("GEETA_OUTBOX_DIR") else ROOT / "outputs" / "email_outbox"

    def send(self, to: str, subject: str, text: str, html: str | None = None,
             attachments: list[Path] | None = None) -> dict:
        msg = EmailMessage()
        sender = self.m.get("from_address") or self.m.get("username") or "geeta@localhost"
        msg["From"] = f'{self.m.get("from_name", "GEETA")} <{sender}>'
        msg["To"] = to
        msg["Subject"] = subject
        msg.set_content(text)
        if html:
            msg.add_alternative(html, subtype="html")
        for a in attachments or []:
            ctype, _ = mimetypes.guess_type(a.name)
            maintype, subtype = (ctype or "application/octet-stream").split("/", 1)
            msg.add_attachment(Path(a).read_bytes(), maintype=maintype, subtype=subtype, filename=Path(a).name)
        if not self.m["ready"]:
            self.outbox.mkdir(parents=True, exist_ok=True)
            self.outbox.chmod(0o700)
            safe = re.sub(r"[^A-Za-z0-9]+", "_", to)[:40]
            p = self.outbox / f"{datetime.now():%Y%m%d-%H%M%S-%f}_{safe}.eml"   # microseconds: no overwrites
            p.write_bytes(bytes(msg))
            p.chmod(0o600)
            return {"status": "outbox", "path": str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p)}
        ctx = ssl.create_default_context()
        port = int(self.m.get("smtp_port") or 587)
        try:
            if port == 465:
                with smtplib.SMTP_SSL(self.m["smtp_host"], port, context=ctx, timeout=30) as s:
                    s.login(self.m["username"], self.m["password"])
                    s.send_message(msg)
            else:
                with smtplib.SMTP(self.m["smtp_host"], port, timeout=30) as s:
                    s.starttls(context=ctx)
                    s.login(self.m["username"], self.m["password"])
                    s.send_message(msg)
            return {"status": "sent"}
        except Exception as e:            # never echo credentials; keep the reason short
            return {"status": "failed", "error": type(e).__name__ + ": " + str(e)[:160]}


# ---------------------------------------------------------------------------
# PDFs (reportlab) - same look as the per-session parent report
# ---------------------------------------------------------------------------
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

INK, ACCENT = colors.HexColor("#3d405b"), colors.HexColor("#e07a5f")


def _styles():
    ss = getSampleStyleSheet()
    return {
        "title": ParagraphStyle("t", parent=ss["Title"], fontSize=20, textColor=INK),
        "h": ParagraphStyle("h", parent=ss["Heading2"], fontSize=13, textColor=ACCENT, spaceBefore=10),
        "body": ParagraphStyle("b", parent=ss["BodyText"], fontSize=11, leading=15),
        "small": ParagraphStyle("s", parent=ss["BodyText"], fontSize=9, textColor=colors.grey),
        "cell": ParagraphStyle("c", parent=ss["BodyText"], fontSize=9.5, leading=12, textColor=INK),
        "head": ParagraphStyle("hd", parent=ss["BodyText"], fontSize=9.5, leading=12, textColor=colors.white,
                               fontName="Helvetica-Bold"),
    }


def _table(rows, widths, st):
    data = [[Paragraph(_x(str(c)), st["head"]) for c in rows[0]]] + \
           [[Paragraph(_x(str(c)), st["cell"]) for c in r] for r in rows[1:]]
    t = Table(data, colWidths=[w * mm for w in widths], repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), INK),
                           ("GRID", (0, 0), (-1, -1), 0.6, colors.HexColor("#dddddd")),
                           ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    return t


def _when(ts: float) -> str:
    return datetime.fromtimestamp(ts).strftime("%-I:%M %p")


MISTAKE_TEXT_FALLBACK = {}


def _mistake_text(cat: str) -> str:
    try:
        from geeta.worksheet import MISTAKE_TEXT
        return MISTAKE_TEXT.get(cat, cat.replace("_", " "))
    except Exception:
        return cat.replace("_", " ")


def daily_parent_pdf(kid: dict, sessions: list[dict], progress: dict, day_label: str, path: Path) -> Path:
    """One PDF for everything a child did since the last nightly email."""
    st = _styles()
    name = kid["nickname"]
    solved = sum(s["solved_count"] or 0 for s in sessions)
    tried = sum(s["tried_count"] or 0 for s in sessions)
    minutes = max(1, round(sum(s["duration_s"] or 0 for s in sessions) / 60))
    hints = sum(s["hints"] or 0 for s in sessions)
    story = [Paragraph(f"{_x(name)}'s maths day", st["title"]),
             Paragraph(f"{_x(day_label)} &nbsp;·&nbsp; GEETA daily report", st["small"]), Spacer(1, 8)]
    headline = (f"{_x(name)} did <b>{len(sessions)} session{'s' if len(sessions) != 1 else ''}</b> "
                f"(about {minutes} minute{'s' if minutes != 1 else ''}) and solved "
                f"<b>{solved} of {tried}</b> question{'s' if tried != 1 else ''} - "
                + ("all by themselves!" if hints == 0 and solved else
                   f"with {hints} hint{'s' if hints != 1 else ''} along the way." if solved else
                   "and kept trying - that's how learning happens."))
    story += [Paragraph("Today at a glance", st["h"]), Paragraph(headline, st["body"])]

    skills: dict[str, dict] = {}
    slips: dict[str, int] = {}
    for s in sessions:
        for sk in s["skills"]:
            d = skills.setdefault(sk["skill"], {"tried": 0, "solved": 0, "hints": 0})
            d["tried"] += sk.get("tried", 0); d["solved"] += sk.get("solved", 0); d["hints"] += sk.get("hints", 0)
        for c, n in (s["mistakes"] or {}).items():
            try:
                from geeta.worksheet import MISTAKE_QUIET
            except Exception:
                MISTAKE_QUIET = set()
            if c not in MISTAKE_QUIET:
                slips[c] = slips.get(c, 0) + n
    if skills:
        story += [Paragraph("What they practised", st["h"]),
                  _table([["Skill", "Solved / tried", "Hints"]] +
                         [[k, f"{v['solved']} / {v['tried']}", v["hints"]] for k, v in skills.items()],
                         [100, 35, 30], st)]
    if slips:
        story += [Paragraph("Worth a little practice together", st["h"]),
                  Paragraph("; ".join(_x(_mistake_text(c)) for c, _ in sorted(slips.items(), key=lambda kv: -kv[1]))
                            + ". These are normal slips - GEETA caught each one and they fixed it themselves.", st["body"])]
    story += [Paragraph("Sessions", st["h"]),
              _table([["Time", "Skills", "Solved", "Grade", "Minutes"]] +
                     [[_when(s["started"]), "; ".join(x["skill"] for x in s["skills"]) or "-",
                       f"{s['solved_count'] or 0} / {s['tried_count'] or 0}", s["grade"] or "-",
                       max(1, round((s["duration_s"] or 0) / 60))] for s in sorted(sessions, key=lambda r: r["started"])],
                     [22, 80, 22, 18, 22], st)]
    if progress.get("skills"):
        story += [Paragraph("Progress so far", st["h"]),
                  Paragraph(f"{progress['sessions']} session(s) with GEETA, {progress['solved']} question(s) solved in total.", st["body"]),
                  _table([["Skill", "Solved / tried", "Where they are"]] +
                         [[s["skill"], f"{s['solved']} / {s['tried']}",
                           {"confident": "Confident", "getting there": "Getting there"}.get(s["status"], "Still learning")]
                          for s in progress["skills"][:8]], [95, 30, 40], st)]
    story += [Spacer(1, 10), Paragraph(
        "GEETA never gives the final answer - it guides, and every answer is checked by computer code. "
        "Only a short learning summary is kept (never the chat, photos or camera). You can switch these emails off "
        "or delete your child's data any time from the Parent page.", st["small"])]
    path.parent.mkdir(parents=True, exist_ok=True)
    SimpleDocTemplate(str(path), pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm,
                      topMargin=16 * mm, bottomMargin=16 * mm).build(story)
    return path


# ---------------------------------------------------------------------------
# Admin technical report
# ---------------------------------------------------------------------------

def _http_ok(url: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


def _services() -> list[list]:
    rows = []
    try:
        out = subprocess.run(["launchctl", "list"], capture_output=True, text=True, timeout=10).stdout
    except Exception:
        out = ""
    for label, desc in (("com.geeta.llama", "Model server"), ("com.geeta.webapp", "Web app"),
                        ("com.geeta.tunnel", "Public tunnel"), ("com.geeta.awake", "Keep-awake"),
                        ("com.geeta.backup", "Nightly backup"), ("com.geeta.nightly", "Nightly reports")):
        line = next((l for l in out.splitlines() if l.endswith(label)), None)
        if line is None:
            rows.append([desc, "NOT INSTALLED", ""])
            continue
        pid, status, _ = line.split("\t")
        running = pid != "-"
        periodic = label in ("com.geeta.backup", "com.geeta.nightly")
        ok = running or (periodic and status == "0")
        rows.append([desc, "OK" if ok else "PROBLEM", f"pid {pid}, last exit {status}"])
    rows.append(["Model answers (:8080/health)", "OK" if _http_ok("http://127.0.0.1:8080/health") else "PROBLEM", ""])
    rows.append(["Web app (:8090/health)", "OK" if _http_ok("http://127.0.0.1:8090/health") else "PROBLEM", ""])
    return rows


def _log_errors() -> list[list]:
    rows = []
    for f in sorted((ROOT / "logs").glob("service_*.log")):
        try:
            txt = f.read_text(errors="ignore")
        except Exception:
            continue
        rows.append([f.name, txt.count("Traceback"), len(re.findall(r"\b(ERROR|CRITICAL)\b", txt)),
                     f"{f.stat().st_size // 1024} KB"])
    return rows


def admin_tech_pdf(store, period_start: float, period_end: float, path: Path, prev: dict | None = None) -> tuple[Path, dict]:
    """Technical report for the admin. Returns (pdf, numbers) - numbers are also stored so the
    next report can show changes (e.g. NEW errors since yesterday)."""
    st = _styles()
    tech = store.tech_between(period_start, period_end)
    usage = store.usage_between(period_start, period_end)
    sec = store.security_between(period_start, period_end)
    services = _services()
    errs = _log_errors()
    backups = sorted((ROOT / "data" / "portal" / "backups").glob("geeta_portal-*.db"))
    du = shutil.disk_usage(str(ROOT))
    db = ROOT / "data" / "portal" / "geeta_portal.db"
    scorecard = {}
    try:
        scorecard = json.load(open(ROOT / "outputs" / "m9" / "scorecard.json"))
    except Exception:
        pass
    url = ""
    try:
        url = subprocess.run([str(ROOT / "deploy" / "public_url.sh")], capture_output=True, text=True, timeout=10).stdout.strip()
    except Exception:
        pass
    total_tb = sum(r[1] for r in errs)
    numbers = {"tracebacks_total": total_tb, **tech, **usage, **sec}
    new_tb = total_tb - (prev or {}).get("tracebacks_total", total_tb)
    problems = [r[0] for r in services if r[1] != "OK"]
    alerts = []
    if problems:
        alerts.append("Service problem: " + ", ".join(problems))
    if new_tb > 0:
        alerts.append(f"{new_tb} new error trace(s) in the service logs since the last report")
    if sec.get("account_locked", 0):
        alerts.append(f"{sec['account_locked']} account lock-out(s) - possible password guessing")
    if du.free / du.total < 0.1:
        alerts.append("Disk is over 90% full")
    if not backups or time.time() - backups[-1].stat().st_mtime > 36 * 3600:
        alerts.append("No database backup in the last 36 hours")
    if tech.get("wrong_answer_risk", 0):
        alerts.append(f"{tech['wrong_answer_risk']} reply/replies blocked by the hallucination gate - review")
    numbers["alerts"] = alerts

    story = [Paragraph("GEETA technical report", st["title"]),
             Paragraph(f"{datetime.fromtimestamp(period_start):%a %d %b %Y %H:%M} to {datetime.fromtimestamp(period_end):%a %d %b %H:%M}"
                       + (f" · {_x(url)}" if url else ""), st["small"]), Spacer(1, 6)]
    story += [Paragraph("Needs attention" if alerts else "All clear", st["h"]),
              Paragraph("<br/>".join("• " + _x(a) for a in alerts) if alerts else
                        "No service problems, no new errors, backups current.", st["body"])]
    story += [Paragraph("Services", st["h"]), _table([["Service", "Status", "Detail"]] + services, [70, 30, 65], st)]
    story += [Paragraph("Usage", st["h"]), _table([["Measure", "Value"]] + [
        ["Tutoring sessions saved", usage["sessions"]], ["Children active", usage["kids_active"]],
        ["Minutes of tutoring", usage["minutes"]], ["Questions tried / solved", f"{usage['tried']} / {usage['solved']}"],
        ["Hints given", usage["hints"]], ["Parent accounts / tutors / children (total)",
                                          f"{usage['parents']} / {usage['tutors']} / {usage['kids_total']}"]], [110, 55], st)]
    story += [Paragraph("Tutoring quality", st["h"]), _table([["Measure", "Value"]] + [
        ["Questions tutored exactly by Python", tech["exact_items"]],
        ["Questions that needed the AI model", tech["model_items"]],
        ["Model replies blocked by the hallucination gate", tech["gate_blocked"]],
        ["Safe fallback replies used", tech["safe_fallbacks"]],
        ["Answer-leak guard blocks", tech["leak_blocks"]],
        ["Verifier checks failed (model working)", tech["verifier_failed"]],
        ["Homework photos read", tech["photos"]],
        ["Latest eval scorecard (M9)", f"{scorecard.get('overall', '?')} / 100"]], [110, 55], st)]
    story += [Paragraph("Security", st["h"]), _table([["Event", "Count"]] + [
        [k.replace("_", " "), v] for k, v in sec.items()] or [["(none)", 0]], [110, 55], st)]
    story += [Paragraph("Logs", st["h"]), _table([["File", "Error traces (all time)", "ERROR lines", "Size"]] + errs, [60, 40, 30, 35], st)]
    story += [Paragraph("Storage and backups", st["h"]), _table([["Item", "Value"]] + [
        ["Disk free", f"{du.free / 1e9:.0f} GB of {du.total / 1e9:.0f} GB"],
        ["Database size", f"{db.stat().st_size // 1024} KB" if db.exists() else "-"],
        ["Latest backup", f"{datetime.fromtimestamp(backups[-1].stat().st_mtime):%d %b %H:%M} ({len(backups)} kept)" if backups else "NONE"]],
        [110, 55], st)]
    story += [Spacer(1, 10), Paragraph("Generated automatically by GEETA. Contains no chat text, photos or children's answers.", st["small"])]
    path.parent.mkdir(parents=True, exist_ok=True)
    SimpleDocTemplate(str(path), pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm,
                      topMargin=16 * mm, bottomMargin=16 * mm).build(story)
    return path, numbers


# ---------------------------------------------------------------------------
# The nightly run
# ---------------------------------------------------------------------------

def run_nightly(store, now: float | None = None, dry_run: bool = False) -> dict:
    """Email each parent a daily PDF for every child active since the last
    report, and every admin the technical report. dry_run: build + outbox only."""
    now = now or time.time()
    settings = load_mail_settings()
    if dry_run:
        settings = dict(settings, ready=False)
    mailer = Mailer(settings)
    day = datetime.fromtimestamp(now).strftime("%A %-d %B %Y")
    out_dir = Path(store.reports_dir) / "daily"      # next to the store it describes (tests stay separate)
    results = {"parents": [], "admin": [], "mode": "send" if settings["ready"] else "outbox"}

    for kid in store.all_kids():
        if not kid.get("parent_id"):
            continue
        parent = store.get_user(kid["parent_id"])
        if not parent or not parent.get("email_reports", True):
            continue
        since = store.last_report_end("parent_daily", kid_id=kid["id"]) or (now - 24 * 3600)
        sessions = store.sessions_between(kid["id"], since, now)
        if not sessions:
            continue
        pdf = daily_parent_pdf(kid, sessions, store.progress(kid["id"]), day,
                               store.kid_report_dir(kid["id"]) / "daily" /
                               f"daily_{datetime.fromtimestamp(now):%Y%m%d}.pdf")
        solved = sum(s["solved_count"] or 0 for s in sessions)
        subject = f"{kid['nickname']}'s maths today - {solved} question{'s' if solved != 1 else ''} solved ⭐"
        text = (f"Hi {parent['name']},\n\n{kid['nickname']} did {len(sessions)} GEETA session(s) today and solved "
                f"{solved} question(s). The full daily report is attached as a PDF.\n\n"
                f"You can see all reports and progress any time on your Parent page.\n"
                f"To stop these emails, switch off \"Email me a daily report\" on the Parent page.\n\n- GEETA")
        html = (f"<p>Hi {_x(parent['name'])},</p><p><b>{_x(kid['nickname'])}</b> did {len(sessions)} GEETA "
                f"session(s) today and solved <b>{solved}</b> question(s). 🎉</p><p>The full daily report is attached.</p>"
                f"<p style='color:#777;font-size:12px'>To stop these emails, switch off \"Email me a daily report\" "
                f"on your Parent page.</p>")
        r = mailer.send(parent["email"], subject, text, html, [pdf])
        store.log_report("parent_daily", since, now, r["status"], kid_id=kid["id"], user_id=parent["id"],
                         detail=json.dumps({k: v for k, v in r.items() if k != "path"}))
        results["parents"].append({"kid": kid["nickname"], "to": parent["email"], **r})

    prev = store.last_report_detail("admin_daily")
    since = store.last_report_end("admin_daily") or (now - 24 * 3600)
    pdf, numbers = admin_tech_pdf(store, since, now, out_dir / f"technical_{datetime.fromtimestamp(now):%Y%m%d}.pdf", prev)
    admins = [u["email"] for u in store.list_users() if u["is_admin"]] + list(settings.get("admin_recipients") or [])
    subject = ("⚠️ GEETA technical report - needs attention" if numbers["alerts"] else "✅ GEETA technical report - all clear")
    text = ("GEETA nightly technical report (PDF attached).\n\n" +
            ("Needs attention:\n- " + "\n- ".join(numbers["alerts"]) if numbers["alerts"] else "All clear.") +
            f"\n\nSessions: {numbers['sessions']}, children active: {numbers['kids_active']}, "
            f"exact-Python questions: {numbers['exact_items']}, model questions: {numbers['model_items']}.")
    statuses = []
    for to in dict.fromkeys(admins):
        r = mailer.send(to, subject, text, None, [pdf])
        statuses.append(r["status"])
        results["admin"].append({"to": to, **r})
    store.log_report("admin_daily", since, now, ",".join(statuses) or "no-admin",
                     detail=json.dumps(numbers, default=str))
    return results


if os.environ.get("GEETA_RUN_NIGHTLY") == "1":
    from geeta.portal_store import PortalStore
    _store = PortalStore(ROOT / "data" / "portal" / "geeta_portal.db", ROOT / "outputs" / "portal")
    print(json.dumps(run_nightly(_store, dry_run=os.environ.get("GEETA_DRY_RUN") == "1"), indent=1, default=str))
